In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd()

if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Не найден корень проекта")

PROCESSED = ROOT / "data" / "processed"

minute_bars = pd.read_csv(
    PROCESSED / "btcusdt_1min_2020_q1.csv"
)

comparison = pd.read_csv(
    PROCESSED / "btcusdt_tr_vpin_with_u2_2020_q1.csv"
)

for column in ["minute_start", "available_at"]:
    minute_bars[column] = pd.to_datetime(
        minute_bars[column], utc=True
    )

comparison["available_at"] = pd.to_datetime(
    comparison["available_at"], utc=True
)

print("Минутных баров:", len(minute_bars))
print("Полных бакетов:", len(comparison))

Минутных баров: 131040
Полных бакетов: 6444


In [16]:
minutes = (
    minute_bars
    .sort_values("available_at")
    .set_index("available_at")
)

assert minutes.index.is_unique
assert minutes.index.is_monotonic_increasing

minute_spacing = minutes.index.to_series().diff().dropna()

assert minute_spacing.eq(pd.Timedelta(minutes=1)).all()

hourly = pd.DataFrame(index=minutes.index)

hourly["volume_prev_1h_btc"] = (
    minutes["volume_btc"]
    .rolling(60, min_periods=60)
    .sum()
)

hourly["empty_minutes_prev_1h"] = (
    minutes["trade_count"].eq(0)
    .astype(int)
    .rolling(60, min_periods=60)
    .sum()
)

hourly["price"] = minutes["close"]

hour_marks = (
    (hourly.index.minute == 0)
    & (hourly.index.second == 0)
)

hourly = hourly.loc[hour_marks].copy()
hourly.index.name = "timestamp"
hourly = hourly.reset_index()

print(hourly.head(5).to_string(index=False))

                timestamp  volume_prev_1h_btc  empty_minutes_prev_1h   price
2020-01-01 01:00:00+00:00          511.814901                    0.0 7177.02
2020-01-01 02:00:00+00:00          883.052603                    0.0 7216.27
2020-01-01 03:00:00+00:00          655.156809                    0.0 7242.85
2020-01-01 04:00:00+00:00          783.724867                    0.0 7225.01
2020-01-01 05:00:00+00:00          467.812578                    0.0 7217.27


In [17]:
signals = (
    comparison
    .dropna(subset=["tr_vpin", "u2_vpin"])
    .sort_values(["available_at", "bucket_id"])
    .drop_duplicates("available_at", keep="last")
    [
        [
            "available_at",
            "bucket_id",
            "tr_vpin",
            "u2_vpin",
        ]
    ]
    .rename(columns={"available_at": "signal_available_at"})
)

hourly_panel = pd.merge_asof(
    hourly.sort_values("timestamp"),
    signals.sort_values("signal_available_at"),
    left_on="timestamp",
    right_on="signal_available_at",
    direction="backward",
    allow_exact_matches=True,
)

hourly_panel["signal_age_minutes"] = (
    hourly_panel["timestamp"]
    - hourly_panel["signal_available_at"]
).dt.total_seconds() / 60

matched = hourly_panel["signal_available_at"].notna()

assert (
    hourly_panel.loc[matched, "signal_available_at"]
    <= hourly_panel.loc[matched, "timestamp"]
).all()

hourly_panel["eligible"] = (
    hourly_panel["tr_vpin"].notna()
    & hourly_panel["u2_vpin"].notna()
    & hourly_panel["volume_prev_1h_btc"].notna()
    & hourly_panel["empty_minutes_prev_1h"].eq(0)
    & hourly_panel["price"].notna()
)

print("Всего часовых моментов:", len(hourly_panel))
print("Пригодно для сравнения:", int(hourly_panel["eligible"].sum()))

print("\nВозраст последнего сигнала, минут:")
print(
    hourly_panel.loc[
        hourly_panel["eligible"], "signal_age_minutes"
    ].describe().round(2).to_string()
)

Всего часовых моментов: 2184
Пригодно для сравнения: 2123

Возраст последнего сигнала, минут:
count    2123.00
mean       17.76
std        16.25
min         0.00
25%         5.00
50%        13.00
75%        25.00
max       134.00


In [18]:
usable = hourly_panel.loc[hourly_panel["eligible"]].copy()

split = pd.Timestamp("2020-03-01", tz="UTC")
test_end = pd.Timestamp("2020-04-01", tz="UTC")

samples = {
    "train_jan_feb": usable.loc[
        usable["timestamp"].lt(split)
    ],
    "test_march": usable.loc[
        usable["timestamp"].ge(split)
        & usable["timestamp"].lt(test_end)
    ],
}

rows = []

for name, frame in samples.items():
    rows.append({
        "sample": name,
        "n_hours": len(frame),
        "corr_tr_u2": frame["tr_vpin"].corr(frame["u2_vpin"]),
        "corr_tr_volume_1h": frame["tr_vpin"].corr(
            frame["volume_prev_1h_btc"]
        ),
        "corr_u2_volume_1h": frame["u2_vpin"].corr(
            frame["volume_prev_1h_btc"]
        ),
    })

hourly_correlations = pd.DataFrame(rows)

print(hourly_correlations.round(6).to_string(index=False))

hourly_panel.to_csv(
    PROCESSED / "hourly_analysis_panel.csv",
    index=False,
)

hourly_correlations.to_csv(
    PROCESSED / "hourly_descriptive_correlations.csv",
    index=False,
)

       sample  n_hours  corr_tr_u2  corr_tr_volume_1h  corr_u2_volume_1h
train_jan_feb     1381    0.905521           0.342650           0.349580
   test_march      741    0.932503           0.748357           0.725591


In [19]:
print(
    hourly_panel.loc[
        hourly_panel["eligible"]
        & hourly_panel["timestamp"].ge(
            pd.Timestamp("2020-04-01", tz="UTC")
        ),
        ["timestamp", "signal_available_at"],
    ].to_string(index=False)
)

                timestamp       signal_available_at
2020-04-01 00:00:00+00:00 2020-03-31 23:59:00+00:00


## Подготовка прогнозной выборки

Прогнозируем реализованную волатильность следующего часа
по минутным логарифмическим доходностям.

Признаки доступны к моменту прогноза.
Будущая волатильность используется только как целевая переменная.

Окна с отсутствующими минутными ценами исключаются.
Цены через остановки торгов не заполняются.

Обучение: январь–февраль.
Проверка: март.
Целевые интервалы обучения не выходят за границу обучения.

In [20]:
minute_features = (
    minute_bars
    .sort_values("available_at")
    .set_index("available_at")
    [["close"]]
    .copy()
)

assert minute_features.index.is_unique

assert (
    minute_features.index.to_series()
    .diff()
    .dropna()
    .eq(pd.Timedelta(minutes=1))
    .all()
)

minute_features["log_return"] = (
    np.log(minute_features["close"]).diff()
)

squared_returns = minute_features["log_return"].pow(2)

minute_features["rv_prev_1h_pct"] = (
    squared_returns
    .rolling(window=60, min_periods=60)
    .sum()
    .pow(0.5)
    .mul(100)
)

minute_features["rv_next_1h_pct"] = (
    minute_features["rv_prev_1h_pct"].shift(-60)
)

print(
    minute_features[
        ["rv_prev_1h_pct", "rv_next_1h_pct"]
    ].dropna().head().to_string()
)

                           rv_prev_1h_pct  rv_next_1h_pct
available_at                                             
2020-01-01 01:01:00+00:00        0.273948        0.429830
2020-01-01 01:02:00+00:00        0.274933        0.429070
2020-01-01 01:03:00+00:00        0.282426        0.424443
2020-01-01 01:04:00+00:00        0.278703        0.424925
2020-01-01 01:05:00+00:00        0.265489        0.425296


In [21]:
check_time = pd.Timestamp("2020-01-03 12:00", tz="UTC")

future_returns = minute_features.loc[
    (minute_features.index > check_time)
    & (
        minute_features.index
        <= check_time + pd.Timedelta(hours=1)
    ),
    "log_return",
]

assert len(future_returns) == 60
assert future_returns.notna().all()

manual_future_rv = (
    100 * np.sqrt(np.sum(future_returns.to_numpy() ** 2))
)

stored_future_rv = minute_features.loc[
    check_time, "rv_next_1h_pct"
]

assert np.isclose(
    manual_future_rv,
    stored_future_rv,
    rtol=1e-12,
    atol=1e-12,
)

print("Начало прогнозного интервала:", check_time)
print("Конец прогнозного интервала:", check_time + pd.Timedelta(hours=1))
print("Прямой расчёт:", manual_future_rv)
print("Значение целевой колонки:", stored_future_rv)
print("Проверка временного сдвига: OK")

Начало прогнозного интервала: 2020-01-03 12:00:00+00:00
Конец прогнозного интервала: 2020-01-03 13:00:00+00:00
Прямой расчёт: 0.4837233761385374
Значение целевой колонки: 0.4837233761385373
Проверка временного сдвига: OK


In [22]:
forecast_panel = hourly_panel.merge(
    minute_features[
        ["rv_prev_1h_pct", "rv_next_1h_pct"]
    ],
    left_on="timestamp",
    right_index=True,
    how="left",
    validate="one_to_one",
)

forecast_panel["target_end"] = (
    forecast_panel["timestamp"] + pd.Timedelta(hours=1)
)

forecast_panel["log_volume_prev_1h"] = np.log1p(
    forecast_panel["volume_prev_1h_btc"]
)

required = [
    "tr_vpin",
    "u2_vpin",
    "log_volume_prev_1h",
    "rv_prev_1h_pct",
    "rv_next_1h_pct",
]

usable_mask = (
    forecast_panel["eligible"]
    & np.isfinite(
        forecast_panel[required].to_numpy(dtype=float)
    ).all(axis=1)
)

model_data = forecast_panel.loc[usable_mask].copy()

split = pd.Timestamp("2020-03-01", tz="UTC")
test_end = pd.Timestamp("2020-04-01", tz="UTC")

train = model_data.loc[
    model_data["timestamp"].lt(split)
    & model_data["target_end"].le(split)
].copy()

test = model_data.loc[
    model_data["timestamp"].ge(split)
    & model_data["target_end"].le(test_end)
].copy()

assert not train.empty
assert not test.empty

assert train["target_end"].max() <= test["timestamp"].min()

assert (
    model_data["signal_available_at"]
    <= model_data["timestamp"]
).all()

for name, frame in [("TRAIN", train), ("TEST", test)]:
    print(f"\n{name}")
    print("Наблюдений:", len(frame))
    print("Первый прогноз:", frame["timestamp"].min())
    print("Последний прогноз:", frame["timestamp"].max())
    print("Последний конец цели:", frame["target_end"].max())
    print(
        "Средняя будущая RV, %:",
        round(frame["rv_next_1h_pct"].mean(), 6),
    )

forecast_panel.to_csv(
    PROCESSED / "hourly_forecast_panel.csv",
    index=False,
)

train.to_csv(PROCESSED / "forecast_train.csv", index=False)
test.to_csv(PROCESSED / "forecast_test.csv", index=False)


TRAIN
Наблюдений: 1378
Первый прогноз: 2020-01-03 03:00:00+00:00
Последний прогноз: 2020-02-29 23:00:00+00:00
Последний конец цели: 2020-03-01 00:00:00+00:00
Средняя будущая RV, %: 0.544515

TEST
Наблюдений: 740
Первый прогноз: 2020-03-01 00:00:00+00:00
Последний прогноз: 2020-03-31 23:00:00+00:00
Последний конец цели: 2020-04-01 00:00:00+00:00
Средняя будущая RV, %: 1.307988


## Первый out-of-sample прогноз

Цель: реализованная волатильность следующего часа.

Все модели оцениваются на январе–феврале.
Коэффициенты фиксируются и применяются к марту.

Сравниваем:
- TR-VPIN отдельно;
- прошлую волатильность и объём;
- контрольные признаки с добавлением TR-VPIN;
- контрольные признаки с добавлением U2-VPIN.

Дополнительные ориентиры:
- средняя целевая волатильность обучения;
- волатильность предыдущего часа.

Первичная оценка: RMSE и MAE на одинаковых тестовых наблюдениях.
Отрицательные прогнозы не обрезаем, но учитываем их количество.

In [23]:
import statsmodels.api as sm

TARGET = "rv_next_1h_pct"

MODEL_FEATURES = {
    "TR_only": [
        "tr_vpin",
    ],
    "Controls": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
    ],
    "Controls_TR": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "tr_vpin",
    ],
    "Controls_U2": [
        "rv_prev_1h_pct",
        "log_volume_prev_1h",
        "u2_vpin",
    ],
}

all_features = sorted({
    feature
    for features in MODEL_FEATURES.values()
    for feature in features
})

for name, frame in [("train", train), ("test", test)]:
    values = frame[all_features + [TARGET]].to_numpy(dtype=float)

    assert np.isfinite(values).all(), f"Некорректные значения в {name}"
    assert frame["timestamp"].is_unique
    assert frame["timestamp"].is_monotonic_increasing

assert train["target_end"].max() <= test["timestamp"].min()

fits = {}

predictions = test[
    ["timestamp", "target_end", TARGET]
].copy()

coefficient_rows = []

for name, features in MODEL_FEATURES.items():
    X_train = sm.add_constant(
        train[features].astype(float),
        has_constant="add",
    )

    X_test = sm.add_constant(
        test[features].astype(float),
        has_constant="add",
    )

    X_test = X_test.reindex(columns=X_train.columns)

    if np.linalg.matrix_rank(X_train.to_numpy()) < X_train.shape[1]:
        raise ValueError(f"{name}: линейно зависимые признаки")

    model = sm.OLS(
        train[TARGET].astype(float),
        X_train,
        missing="raise",
    ).fit()

    fits[name] = model
    predictions[name] = model.predict(X_test)

    for term, coefficient in model.params.items():
        coefficient_rows.append({
            "model": name,
            "term": term,
            "coefficient": float(coefficient),
        })

predictions["Train_mean"] = float(train[TARGET].mean())
predictions["Persistence"] = test["rv_prev_1h_pct"].to_numpy()

coefficients = pd.DataFrame(coefficient_rows)

print("Обучение завершено.")
print("Тестовых прогнозов на модель:", len(predictions))

print("\nКОЭФФИЦИЕНТЫ:")
print(coefficients.round(6).to_string(index=False))

Обучение завершено.
Тестовых прогнозов на модель: 740

КОЭФФИЦИЕНТЫ:
      model               term  coefficient
    TR_only              const     0.179234
    TR_only            tr_vpin     1.514336
   Controls              const    -0.697074
   Controls     rv_prev_1h_pct     0.210037
   Controls log_volume_prev_1h     0.149010
Controls_TR              const    -0.730808
Controls_TR     rv_prev_1h_pct     0.197493
Controls_TR log_volume_prev_1h     0.141424
Controls_TR            tr_vpin     0.406077
Controls_U2              const    -0.746560
Controls_U2     rv_prev_1h_pct     0.199153
Controls_U2 log_volume_prev_1h     0.141073
Controls_U2            u2_vpin     0.417136


In [24]:
prediction_columns = [
    *MODEL_FEATURES.keys(),
    "Train_mean",
    "Persistence",
]

actual = predictions[TARGET].to_numpy(dtype=float)

metric_rows = []

for name in prediction_columns:
    forecast = predictions[name].to_numpy(dtype=float)

    assert np.isfinite(forecast).all()

    error = forecast - actual

    metric_rows.append({
        "model": name,
        "n_test": len(actual),
        "MSE": float(np.mean(error ** 2)),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "negative_predictions": int((forecast < 0).sum()),
    })

metrics = pd.DataFrame(metric_rows).set_index("model")

controls_mse = metrics.loc["Controls", "MSE"]

if controls_mse <= 0:
    raise ValueError("Нельзя рассчитать относительную ошибку")

metrics["MSE_reduction_vs_controls_pct"] = (
    100 * (1 - metrics["MSE"] / controls_mse)
)

print(
    metrics.sort_values("RMSE")
    .round(6)
    .to_string()
)

             n_test       MSE      RMSE       MAE      bias  negative_predictions  MSE_reduction_vs_controls_pct
model                                                                                                           
Persistence     740  1.423754  1.193212  0.487237  0.000281                     0                      20.174700
Controls_U2     740  1.752057  1.323653  0.562291 -0.485128                     0                       1.767804
Controls_TR     740  1.770869  1.330740  0.570727 -0.497050                     0                       0.713065
Controls        740  1.783587  1.335510  0.576553 -0.502451                     0                       0.000000
TR_only         740  2.448769  1.564854  0.743873 -0.688507                     0                     -37.294646
Train_mean      740  2.746281  1.657191  0.823197 -0.763473                     0                     -53.975149


In [25]:
import json

TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    PROCESSED / "hourly_test_predictions_baseline.csv",
    index=False,
)

metrics.to_csv(
    TABLES / "baseline_forecast_metrics.csv",
)

coefficients.to_csv(
    TABLES / "baseline_regression_coefficients.csv",
    index=False,
)

experiment_config = {
    "target": TARGET,
    "forecast_horizon_minutes": 60,
    "models": MODEL_FEATURES,
    "fit_method": "OLS with intercept",
    "training": "January–February 2020",
    "testing": "March 2020",
    "refit_during_test": False,
    "negative_forecasts_clipped": False,
    "inference": "not yet performed",
}

with (ROOT / "baseline_forecast_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Прогнозы, метрики, коэффициенты и настройки сохранены.")

Прогнозы, метрики, коэффициенты и настройки сохранены.


In [26]:
event_start = pd.Timestamp("2020-03-12", tz="UTC")
event_end = pd.Timestamp("2020-03-14", tz="UTC")

event_target = (
    predictions["timestamp"].ge(event_start)
    & predictions["target_end"].le(event_end)
)

outside_event_target = (
    predictions["target_end"].le(event_start)
    | predictions["timestamp"].ge(event_end)
)

crossing_boundary = ~(event_target | outside_event_target)

periods = {
    "all_march": predictions,
    "event_target_hours": predictions.loc[event_target],
    "other_target_hours": predictions.loc[outside_event_target],
}

print("Целевые часы внутри события:", int(event_target.sum()))
print("Целевые часы вне события:", int(outside_event_target.sum()))
print(
    "Часы, пересекающие границу события:",
    int(crossing_boundary.sum()),
)

Целевые часы внутри события: 48
Целевые часы вне события: 692
Часы, пересекающие границу события: 0


In [27]:
period_rows = []

for period_name, frame in periods.items():
    if frame.empty:
        raise ValueError(f"Пустой период: {period_name}")

    actual = frame[TARGET].to_numpy(dtype=float)

    controls_error = (
        frame["Controls"].to_numpy(dtype=float) - actual
    )
    persistence_error = (
        frame["Persistence"].to_numpy(dtype=float) - actual
    )

    controls_mse = np.mean(controls_error ** 2)
    persistence_mse = np.mean(persistence_error ** 2)

    for model_name in prediction_columns:
        forecast = frame[model_name].to_numpy(dtype=float)
        error = forecast - actual
        mse = np.mean(error ** 2)

        period_rows.append({
            "period": period_name,
            "model": model_name,
            "n_hours": len(frame),
            "RMSE": float(np.sqrt(mse)),
            "MAE": float(np.mean(np.abs(error))),
            "bias": float(np.mean(error)),
            "MSE_gain_vs_controls_pct": (
                float(100 * (1 - mse / controls_mse))
                if controls_mse > 0 else np.nan
            ),
            "MSE_gain_vs_persistence_pct": (
                float(100 * (1 - mse / persistence_mse))
                if persistence_mse > 0 else np.nan
            ),
        })

period_metrics = pd.DataFrame(period_rows)

for period_name in periods:
    print(f"\n{period_name.upper()}")

    print(
        period_metrics.loc[
            period_metrics["period"].eq(period_name)
        ]
        .drop(columns="period")
        .sort_values("RMSE")
        .round(6)
        .to_string(index=False)
    )

period_metrics.to_csv(
    TABLES / "baseline_metrics_by_event_window.csv",
    index=False,
)


ALL_MARCH
      model  n_hours     RMSE      MAE      bias  MSE_gain_vs_controls_pct  MSE_gain_vs_persistence_pct
Persistence      740 1.193212 0.487237  0.000281                 20.174700                     0.000000
Controls_U2      740 1.323653 0.562291 -0.485128                  1.767804                   -23.058976
Controls_TR      740 1.330740 0.570727 -0.497050                  0.713065                   -24.380284
   Controls      740 1.335510 0.576553 -0.502451                  0.000000                   -25.273567
    TR_only      740 1.564854 0.743873 -0.688507                -37.294646                   -71.993900
 Train_mean      740 1.657191 0.823197 -0.763473                -53.975149                   -92.890161

EVENT_TARGET_HOURS
      model  n_hours     RMSE      MAE      bias  MSE_gain_vs_controls_pct  MSE_gain_vs_persistence_pct
Persistence       48 4.062220 2.275547 -0.049782                 16.748241                     0.000000
Controls_U2       48 4.430960 2.7

## Промежуточный результат

На отложенной выборке марта 2020:
- добавление TR-VPIN к контрольной регрессии снизило MSE на 0.71%;
- добавление U2-VPIN снизило MSE на 1.77%;
- лучшим из рассмотренных прогнозов оказался Persistence;
- самостоятельная модель TR-VPIN уступила контрольной регрессии.

Результат относится к выбранной спецификации и периоду.
Статистическая неопределённость различий пока не оценена.

Разбиение на событийные и остальные часы используется
для диагностики, а не для выбора основной тестовой выборки.

## Диагностика результатов по событийным часам

На отложенном марте 2020 добавление TR-VPIN к контрольной
регрессии снижает MSE на 0.71%, а добавление U2-VPIN — на 1.77%.

Направление результата сохраняется:
- в целевых часах 12–13 марта;
- в остальных целевых часах марта.

Во всех трёх сравнениях Persistence превосходит
рассмотренные регрессионные модели по RMSE и MAE.

Следовательно, эффект не ограничивается только двумя днями обвала.
Однако размеры улучшений и их статистическая неопределённость
требуют отдельной проверки.

Результаты относятся к BTCUSDT, выбранному периоду,
часовому горизонту и фиксированным спецификациям.